# Práctica — Clase 6: Entrenar una CNN

Curso **Deep Learning con Python y PyTorch desde cero** · DIH DATAlife (Hub de Innovación Digital Europeo)

En esta práctica trabajas sobre el mismo problema del notebook de teoría (entrenar una red neuronal convolucional, o CNN, sobre CIFAR-10) para completar tú mismo las partes clave del proceso. Hay dos ejercicios:

1. Completar el bucle de entrenamiento de una CNN ya definida
2. Calcular e imprimir la exactitud (accuracy) del modelo entrenado, en global y por clase

Completa las partes marcadas con `# TODO:` antes de mirar la solución. Ejecuta primero la celda de preparación, que es común a los dos ejercicios.

## Preparación común

Esta celda carga CIFAR-10 con `torchvision`, lo normaliza y lo deja listo en `DataLoader`, y define la misma arquitectura de CNN que usaste en el notebook de teoría (adaptada a 3 canales de entrada). Ejecútala antes de empezar cualquiera de los dos ejercicios.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

print('Versión de PyTorch:', torch.__version__)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class_names = ['avión', 'automóvil', 'pájaro', 'gato', 'ciervo',
               'perro', 'rana', 'caballo', 'barco', 'camión']


class ConvNetCIFAR(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model = ConvNetCIFAR()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print('Modelo listo. Imágenes de entrenamiento:', len(train_dataset))

## Ejercicio 1 · Completa el bucle de entrenamiento

**Enunciado:** en el notebook de teoría viste el bucle de entrenamiento completo de una CNN: forward, cálculo de la pérdida, backward y actualización de pesos. Aquí tienes el mismo bucle, pero con tres líneas clave sustituidas por `# TODO:`. Complétalas para que el modelo entrene correctamente durante 2 épocas.

**Pistas:**

- Antes de calcular la pérdida de un nuevo lote, tienes que limpiar los gradientes acumulados del lote anterior. ¿Qué método del optimizador hace eso?
- Una vez calculada la pérdida, ¿qué método de `loss` calcula los gradientes hacia atrás (retropropagación)?
- Por último, ¿qué método del optimizador usa esos gradientes para actualizar los pesos del modelo?
- Usamos solo 2 épocas en este ejercicio para que la práctica no tarde demasiado; en el notebook de teoría usaste 5

In [ ]:
epocas = 2

for epoca in range(epocas):
    model.train()
    perdida_acumulada = 0.0
    for imagenes, etiquetas in train_loader:
        optimizer.zero_grad()          # limpiamos los gradientes del lote anterior
        salidas = model(imagenes)      # forward
        loss = criterion(salidas, etiquetas)

        # TODO: calcula los gradientes de la pérdida (retropropagación)

        # TODO: actualiza los pesos del modelo usando el optimizador

        perdida_acumulada += loss.item()

    perdida_media = perdida_acumulada / len(train_loader)
    print(f'Época {epoca + 1}/{epocas} - pérdida media: {perdida_media:.4f}')

<details>
<summary>💡 Ver solución</summary>

```python
epocas = 2

for epoca in range(epocas):
    model.train()
    perdida_acumulada = 0.0
    for imagenes, etiquetas in train_loader:
        optimizer.zero_grad()          # limpiamos los gradientes del lote anterior
        salidas = model(imagenes)      # forward
        loss = criterion(salidas, etiquetas)

        loss.backward()                # calculamos los gradientes (retropropagación)
        optimizer.step()               # actualizamos los pesos del modelo

        perdida_acumulada += loss.item()

    perdida_media = perdida_acumulada / len(train_loader)
    print(f'Época {epoca + 1}/{epocas} - pérdida media: {perdida_media:.4f}')

# La pérdida media debería bajar de una época a la siguiente: es la señal de
# que el modelo está aprendiendo. Con solo 2 épocas no esperes una exactitud
# muy alta todavía; en el notebook de teoría, con 5 épocas, se nota mejor.
```
</details>

## Ejercicio 2 · Calcula la exactitud del modelo entrenado

**Enunciado:** con el modelo ya entrenado en el ejercicio anterior, calcula la exactitud (accuracy) sobre el conjunto de prueba de CIFAR-10, primero en global y después clase por clase (¿el modelo distingue mejor los perros que los gatos, por ejemplo?).

**Pistas:**

- Recuerda poner el modelo en modo evaluación con `model.eval()` y desactivar el cálculo de gradientes con `torch.no_grad()`, porque no vas a entrenar nada en esta celda
- `torch.argmax(salidas, dim=1)` te da la clase predicha para cada imagen del lote
- Para la exactitud por clase, necesitas dos contadores por cada una de las diez clases: cuántas veces esa clase apareció (`total_por_clase`) y cuántas veces el modelo la predijo correctamente (`correctas_por_clase`). Una forma sencilla es usar dos listas de diez ceros y recorrer el lote con un bucle `for`
- La exactitud de una clase es `correctas_por_clase[i] / total_por_clase[i]`

In [ ]:
model.eval()

correctas = 0
total = 0
correctas_por_clase = [0] * 10
total_por_clase = [0] * 10

with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        salidas = model(imagenes)
        predicciones = torch.argmax(salidas, dim=1)

        # TODO: actualiza "correctas" y "total" para la exactitud global

        # TODO: recorre el lote (con un for sobre range(len(etiquetas))) y
        #       actualiza correctas_por_clase[...] y total_por_clase[...]
        #       para la exactitud por clase
        pass

# TODO: imprime la exactitud global: correctas / total

# TODO: imprime la exactitud de cada una de las diez clases, usando
#       class_names[i] para el nombre y correctas_por_clase[i] / total_por_clase[i]
#       para el valor (cuidado con dividir entre cero si alguna clase no
#       aparece en el conjunto de prueba)


<details>
<summary>💡 Ver solución</summary>

```python
model.eval()

correctas = 0
total = 0
correctas_por_clase = [0] * 10
total_por_clase = [0] * 10

with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        salidas = model(imagenes)
        predicciones = torch.argmax(salidas, dim=1)

        correctas += (predicciones == etiquetas).sum().item()
        total += etiquetas.size(0)

        for i in range(len(etiquetas)):
            etiqueta_real = etiquetas[i].item()
            total_por_clase[etiqueta_real] += 1
            if predicciones[i] == etiquetas[i]:
                correctas_por_clase[etiqueta_real] += 1

print(f'Exactitud global: {correctas / total:.2%}')
print()
for i in range(10):
    if total_por_clase[i] > 0:
        exactitud_clase = correctas_por_clase[i] / total_por_clase[i]
        print(f'  {class_names[i]:>10}: {exactitud_clase:.2%}')

# Con solo 2 épocas de entrenamiento es normal que la exactitud todavía sea
# modesta y que varíe bastante entre clases: algunas clases (por ejemplo,
# barco o camión) suelen ser más fáciles de distinguir que otras (gato y
# perro, con formas y texturas más parecidas entre sí).
```
</details>

## Para seguir practicando

Si quieres ir un poco más allá:

- Vuelve al notebook de teoría, entrena el mismo modelo durante 5 épocas en vez de 2, y compara la exactitud global y por clase con la que obtuviste aquí
- Retoma el cálculo del capítulo 6 del manual: compara el número de parámetros de `model.conv1` con los que necesitaría la primera capa de una red densa equivalente, y anota tu propia conclusión sobre por qué la CNN es más eficiente

Cuando termines, vuelve al capítulo 6 del manual y contrasta tus resultados con el resumen de puntos clave: ¿coinciden tus conclusiones con lo que predice la teoría?